<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-research/perch-hoplite/blob/main/perch_hoplite/agile/1_embed_audio_v2.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-research/perch-hoplite/blob/main/perch_hoplite/agile/1_embed_audio_v2.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# Overview

This notebook uses perch-hoplite to compute and save embeddings for set of audio files using a pre-trained model. This is the first step in the agile modeling process. If the data you wish to search and classify is already embedded with a pre-trained model into a perch-hoplite database, then proceed to the step 2 colab notebook ([2_agile_modeling_v2.ipynb](https://github.com/google-research/perch-hoplite/blob/main/perch_hoplite/agile/2_agile_modeling_v2.ipynb)).

## [Optional] perch-hoplite installation for hosted runtimes

If you have not already installed perch-hoplite (particularly if you are using a hosted Colab runtime), make sure to install perch-hoplite from the Github source to ensure the most recent version is installed. After installation, you will need to restart your runtime before running anything else. Go to the top menu, select "Runtime" then "Restart Session".

**If you want to use the Perch V2 model, you must be on a GPU runtime and additionally install TensorFlow version 2.20.rc0 (experimental).**

In [1]:
# PRE PROCESSING STEP 1: TIME EXPANSION OF DATA AND SAMPLE RATE NORMALISATION
import os
import soundfile as sf
import librosa

IN_DIR  = r"C:\Users\Administrator\hello\uk_bats\uk_audio"
OUT_DIR = r"C:\Users\Administrator\hello\uk_bats\uk_audio\uk_audio_timeexp10"
os.makedirs(OUT_DIR, exist_ok=True)

EXPANSION_FACTOR = 10
TARGET_SR = 384000

for root, _, files in os.walk(IN_DIR):

    if os.path.abspath(root) == os.path.abspath(OUT_DIR):
        continue

    for fname in files:
        if not fname.lower().endswith(".wav"):
            continue

        in_path = os.path.join(root, fname)

        data, sr = sf.read(in_path)

        if data.ndim > 1:
            data = data.mean(axis=1)

        if sr != TARGET_SR:
            data = librosa.resample(data, orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR

        new_sr = int(sr / EXPANSION_FACTOR)

        # ✅ KEEP ORIGINAL FILENAME
        out_path = os.path.join(OUT_DIR, fname)

        sf.write(out_path, data, new_sr)


In [2]:
#CHECK STEP 1 COMPLETE 
import os

out_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".wav")]

print("Time-expanded WAV files created:", len(out_files))

# Show a few example filenames
out_files[:10]


Time-expanded WAV files created: 9997


['00000_myotis_alcathoe_0_1.0.wav',
 '00001_myotis_alcathoe_0_1.0.wav',
 '00002_myotis_alcathoe_0_1.0.wav',
 '00003_myotis_alcathoe_0_1.0.wav',
 '00004_myotis_alcathoe_0_1.0.wav',
 '00005_barbastellus_barbastellus_0_1.0.wav',
 '00006_barbastellus_barbastellus_0_1.0.wav',
 '00007_barbastellus_barbastellus_0_1.0.wav',
 '00008_barbastellus_barbastellus_0_1.0.wav',
 '00009_barbastellus_barbastellus_0_1.0.wav']

In [3]:
import sys
from pathlib import Path
import pandas as pd

SOURCE_DIR = Path(r"C:\Users\Administrator\hello\uk_bats\uk_audio\uk_audio_timeexp10")
CSV_PATH   = Path(r"C:\Users\Administrator\hello\uk_bats\uk_same_test_annotations.csv")

print("Python:", sys.version.replace("\n"," "))
print("pandas:", pd.__version__)
print("SOURCE_DIR exists:", SOURCE_DIR.exists(), "| is_dir:", SOURCE_DIR.is_dir())
print("CSV exists:", CSV_PATH.exists())

# --- Read CSV robustly as strings ---
df = pd.read_csv(CSV_PATH, dtype={"recording_id": "string"})
print("\nCSV columns:", df.columns.tolist())
print("recording_id dtype:", df["recording_id"].dtype)
print("rows:", len(df), "| unique recording_id:", df["recording_id"].nunique())
print("example recording_id:", df["recording_id"].dropna().unique()[:5])

# --- Scan wavs (recursive) ---
wav_paths = [p for p in SOURCE_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".wav"]
print("\nFound .wav files under SOURCE_DIR:", len(wav_paths))
print("Example wav basenames:", [p.name for p in wav_paths[:10]])
print("Example wav relpaths:", [str(p.relative_to(SOURCE_DIR)) for p in wav_paths[:5]])

# --- Compare matching modes ---
exclude_full = set(df["recording_id"].dropna().astype(str).str.strip().str.lower())
exclude_stem = set(Path(x).stem.strip().lower() for x in df["recording_id"].dropna().astype(str))

source_full = set(p.name.lower() for p in wav_paths)
source_stem = set(p.stem.lower() for p in wav_paths)

full_matches = source_full & exclude_full
stem_matches = source_stem & exclude_stem

print("\nExact-filename matches (including .wav):", len(full_matches))
print("Example exact matches:", list(sorted(full_matches))[:10])

print("\nStem-only matches (without extension):", len(stem_matches))
print("Example stem matches:", list(sorted(stem_matches))[:10])

# --- If both are zero, test substring relationship quickly ---
if len(full_matches) == 0 and len(stem_matches) == 0:
    sample_files = [p.name.lower() for p in wav_paths[:1000]]
    sample_ids = list(exclude_stem)[:200]
    hit = [rid for rid in sample_ids if any(rid in fn for fn in sample_files)]
    print("\nSubstring hits among samples:", len(hit))
    print("Example substring hits:", hit[:10])


Python: 3.11.14 (main, Oct 31 2025, 22:57:10) [MSC v.1944 64 bit (AMD64)]
pandas: 2.3.3
SOURCE_DIR exists: True | is_dir: True
CSV exists: True

CSV columns: ['Unnamed: 0', 'recording_id', 'end_time', 'start_time', 'low_freq', 'high_freq', 'event', 'class', 'recording_path']
recording_id dtype: string
rows: 4541 | unique recording_id: 369
example recording_id: <StringArray>
['15060010_0_1.0.wav',  'eb00021_0_1.0.wav',  'eb00074_0_1.0.wav',
  'eb00126_0_1.0.wav',  'eb00144_0_1.0.wav']
Length: 5, dtype: string

Found .wav files under SOURCE_DIR: 9997
Example wav basenames: ['00000_myotis_alcathoe_0_1.0.wav', '00001_myotis_alcathoe_0_1.0.wav', '00002_myotis_alcathoe_0_1.0.wav', '00003_myotis_alcathoe_0_1.0.wav', '00004_myotis_alcathoe_0_1.0.wav', '00005_barbastellus_barbastellus_0_1.0.wav', '00006_barbastellus_barbastellus_0_1.0.wav', '00007_barbastellus_barbastellus_0_1.0.wav', '00008_barbastellus_barbastellus_0_1.0.wav', '00009_barbastellus_barbastellus_0_1.0.wav']
Example wav relpaths:

In [5]:
#PREPROCESSING STEP 2:EXCLUDE CLIPS FROM TEST DATASET

import shutil
from pathlib import Path, PurePosixPath
from collections import defaultdict, Counter
import pandas as pd

# ---------------------------
# CONFIG
# ---------------------------
SOURCE_DIR = Path(r"C:\Users\Administrator\hello\uk_bats\uk_audio\uk_audio_timeexp10")  # flat (9997)
DEST_DIR   = Path(r"C:\Users\Administrator\hello\uk_bats\uk_audio\uk_audio_timeexp10_train")

TRAIN_CSV  = Path(r"C:\Users\Administrator\hello\uk_bats\uk_same_train_annotations.csv")
TEST_CSV   = Path(r"C:\Users\Administrator\hello\uk_bats\uk_same_test_annotations.csv")

UK_AUDIO_ROOT = SOURCE_DIR.parent  # ...\uk_audio (contains bat_data_2018/, etc.)

AUDIO_EXT = ".wav"
HARD_RESET_DEST_DIR = True
DRY_RUN = False

# Expected split from your screenshot
EXPECTED_TEST  = 369
EXPECTED_TRAIN = 7010
EXPECTED_TOTAL = EXPECTED_TEST + EXPECTED_TRAIN  # 7379

# This is the root with almost all empty clips (per your log)
EMPTY_HEAVY_ROOT = "bat_detective_batdetect2"

# ---------------------------
# Helpers
# ---------------------------
def canon_posix(p: str) -> str:
    s = str(p).strip().replace("\\", "/")
    s = s.lstrip("./").lstrip("/")
    return PurePosixPath(s).as_posix()

def canon_basename(x) -> str:
    return Path(str(x)).name.strip().lower()

def root_from_recording_path(rp: str) -> str:
    parts = PurePosixPath(canon_posix(rp)).parts
    return parts[0] if parts else ""

def is_within(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False

# ---------------------------
# Safety checks
# ---------------------------
if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"SOURCE_DIR not found: {SOURCE_DIR}")
if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"TRAIN_CSV not found: {TRAIN_CSV}")
if not TEST_CSV.exists():
    raise FileNotFoundError(f"TEST_CSV not found: {TEST_CSV}")

if SOURCE_DIR.resolve() == DEST_DIR.resolve():
    raise ValueError("DEST_DIR must be different from SOURCE_DIR.")
if is_within(DEST_DIR, SOURCE_DIR) or is_within(SOURCE_DIR, DEST_DIR):
    raise ValueError("Refusing to copy because SOURCE_DIR and DEST_DIR are nested (would recurse).")

# ---------------------------
# Load CSVs
# ---------------------------
train_df = pd.read_csv(TRAIN_CSV, dtype={"recording_id": "string", "recording_path": "string"})
test_df  = pd.read_csv(TEST_CSV,  dtype={"recording_id": "string", "recording_path": "string"})

required = {"recording_id", "recording_path", "event", "class"}
for name, df in [("TRAIN_CSV", train_df), ("TEST_CSV", test_df)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

train_roots = sorted({root_from_recording_path(x) for x in train_df["recording_path"].dropna().unique()})
test_roots  = sorted({root_from_recording_path(x) for x in test_df["recording_path"].dropna().unique()})
allowed_roots = sorted(set(train_roots) | set(test_roots))

print("Allowed dataset roots (from CSV recording_path):")
for r in allowed_roots:
    print("  -", r)

# ---------------------------
# Scan SOURCE_DIR (flat)
# ---------------------------
source_wavs = [p for p in SOURCE_DIR.rglob("*") if p.is_file() and p.suffix.lower() == AUDIO_EXT]
source_map = {}
for p in source_wavs:
    k = p.name.lower()
    if k in source_map:
        raise RuntimeError(f"Duplicate filename inside SOURCE_DIR (flat collision): {p.name}")
    source_map[k] = p
source_names = set(source_map.keys())

print(f"\nTotal WAVs found in SOURCE_DIR (flat): {len(source_wavs)}")

# ---------------------------
# Map flat filenames -> roots by scanning UK_AUDIO_ROOT/<root>/audio/**.wav (recursive)
# ---------------------------
name_to_root = {}
name_to_fullpath = {}
collisions = defaultdict(list)

for root in allowed_roots:
    root_dir = UK_AUDIO_ROOT / root
    if not root_dir.is_dir():
        raise FileNotFoundError(f"Expected root folder missing: {root_dir}")

    audio_dir = root_dir / "audio"
    search_dir = audio_dir if audio_dir.is_dir() else root_dir

    for p in search_dir.rglob("*"):
        if not p.is_file() or p.suffix.lower() != AUDIO_EXT:
            continue
        bn = p.name.lower()
        collisions[bn].append((root, str(p)))

bad = {bn: locs for bn, locs in collisions.items() if len(locs) > 1}
if bad:
    ex_bn, ex_locs = next(iter(bad.items()))
    print(f"\nERROR: {len(bad)} basenames occur in multiple roots. Flat folder cannot disambiguate.")
    print("Example:", ex_bn)
    for r, path in ex_locs[:10]:
        print("  -", r, "->", path)
    raise RuntimeError("Resolve duplicate basenames (or preserve folder structure) before proceeding.")

for bn, locs in collisions.items():
    root, fullpath = locs[0]
    name_to_root[bn] = root
    name_to_fullpath[bn] = fullpath

eligible_in_source = set(name_to_root.keys()) & source_names

print(f"\nWAVs in SOURCE_DIR that map to allowed roots: {len(eligible_in_source)}")
print(f"WAVs in SOURCE_DIR outside allowed roots:     {len(source_names - eligible_in_source)}")

# ---------------------------
# Annotated sets
# ---------------------------
train_annotated = set(train_df["recording_id"].dropna().map(canon_basename).unique())
test_annotated  = set(test_df["recording_id"].dropna().map(canon_basename).unique())
annotated_total = train_annotated | test_annotated

print("\nUnique recordings with >=1 annotated event:")
print(f"  TRAIN_CSV: {len(train_annotated)}")
print(f"  TEST_CSV:  {len(test_annotated)}")
print(f"  TOTAL:     {len(annotated_total)}")

# Ensure test files exist in SOURCE_DIR
missing_test = test_annotated - source_names
if missing_test:
    print(f"\nERROR: {len(missing_test)} test recording_id WAVs not found in SOURCE_DIR.")
    print("Examples:", sorted(list(missing_test))[:20])
    raise RuntimeError("Test WAVs missing from SOURCE_DIR (name mismatch or different source folder).")

# No-event candidates are eligible but absent from BOTH CSVs
no_event_candidates = eligible_in_source - annotated_total
print(f"\nNo-event candidates (eligible but not in CSVs): {len(no_event_candidates)}")

no_event_by_root = Counter(name_to_root[n] for n in no_event_candidates)
print("\nNo-event candidates by root (top):")
for r, c in no_event_by_root.most_common(30):
    print(f"  {r}: {c}")

# ---------------------------
# TARGET: match screenshot totals
# ---------------------------
target_no_event = EXPECTED_TOTAL - len(annotated_total)
if target_no_event < 0:
    raise RuntimeError(f"Expected total ({EXPECTED_TOTAL}) is smaller than annotated_total ({len(annotated_total)}).")

print(f"\nTarget no-event count (to reach TOTAL={EXPECTED_TOTAL}): {target_no_event}")

# Keep ALL no-event from roots other than EMPTY_HEAVY_ROOT, then trim that root deterministically
no_event_non_heavy = sorted([n for n in no_event_candidates if name_to_root[n] != EMPTY_HEAVY_ROOT],
                            key=lambda n: name_to_fullpath.get(n, n))
no_event_heavy = sorted([n for n in no_event_candidates if name_to_root[n] == EMPTY_HEAVY_ROOT],
                        key=lambda n: name_to_fullpath.get(n, n))

need_from_heavy = target_no_event - len(no_event_non_heavy)
if need_from_heavy < 0:
    # too many no-event in non-heavy roots; trim them too (rare)
    print("\nWARNING: non-heavy no-event exceed target; trimming non-heavy deterministically.")
    keep_no_event = no_event_non_heavy[:target_no_event]
else:
    if need_from_heavy > len(no_event_heavy):
        raise RuntimeError(
            f"Not enough no-event files in {EMPTY_HEAVY_ROOT} to reach target. "
            f"Need {need_from_heavy}, have {len(no_event_heavy)}."
        )
    keep_no_event = no_event_non_heavy + no_event_heavy[:need_from_heavy]

keep_no_event = set(keep_no_event)
dropped_no_event = sorted(list(no_event_candidates - keep_no_event), key=lambda n: name_to_fullpath.get(n, n))

# Universe = all annotated + selected no-event
universe = (annotated_total & eligible_in_source) | keep_no_event

# Split within universe
test_eff = test_annotated & universe
train_eff = sorted(list(universe - test_eff))

print("\nFINAL COUNTS:")
print(f"  Universe TOTAL: {len(universe)} (expected {EXPECTED_TOTAL})")
print(f"  TEST:           {len(test_eff)} (expected {EXPECTED_TEST})")
print(f"  TRAIN:          {len(train_eff)} (expected {EXPECTED_TRAIN})")
print(f"  Dropped no-event (excess): {len(dropped_no_event)}")

if len(universe) != EXPECTED_TOTAL or len(test_eff) != EXPECTED_TEST or len(train_eff) != EXPECTED_TRAIN:
    raise AssertionError("Final counts still mismatch expected totals.")

# ---------------------------
# Copy TRAIN WAVs
# ---------------------------
if DRY_RUN:
    print("\nDRY_RUN=True → not copying anything.")
else:
    if HARD_RESET_DEST_DIR and DEST_DIR.exists():
        shutil.rmtree(DEST_DIR)
    DEST_DIR.mkdir(parents=True, exist_ok=True)

    copied = 0
    for name in train_eff:
        src = source_map[name]
        dst = DEST_DIR / src.name
        shutil.copy2(src, dst)
        copied += 1

    print(f"\nDONE: copied TRAIN WAVs: {copied} → {DEST_DIR}")

# ---------------------------
# Manifests
# ---------------------------
pd.DataFrame({"filename": train_eff}).to_csv(DEST_DIR / "train_files_manifest.csv", index=False)
pd.DataFrame({"filename": sorted(list(test_eff))}).to_csv(DEST_DIR / "test_files_manifest.csv", index=False)
pd.DataFrame({"filename": sorted(list(keep_no_event))}).to_csv(DEST_DIR / "kept_no_event_files.csv", index=False)
pd.DataFrame({"filename": dropped_no_event}).to_csv(DEST_DIR / "dropped_excess_no_event_files.csv", index=False)

print("\nWrote manifests:")
print(f"  - {DEST_DIR / 'train_files_manifest.csv'}")
print(f"  - {DEST_DIR / 'test_files_manifest.csv'}")
print(f"  - {DEST_DIR / 'kept_no_event_files.csv'}")
print(f"  - {DEST_DIR / 'dropped_excess_no_event_files.csv'}")



Allowed dataset roots (from CSV recording_path):
  - bat_data_2018
  - bat_data_2018_test
  - bat_data_2019
  - bat_data_2019_test
  - bat_detective_batdetect2
  - bcireland
  - bct_1_sec
  - echobank_batdetect2
  - rhinolophus_bct
  - sn_scot_nor

Total WAVs found in SOURCE_DIR (flat): 9997

WAVs in SOURCE_DIR that map to allowed roots: 7564
WAVs in SOURCE_DIR outside allowed roots:     2433

Unique recordings with >=1 annotated event:
  TRAIN_CSV: 5385
  TEST_CSV:  369
  TOTAL:     5754

No-event candidates (eligible but not in CSVs): 1810

No-event candidates by root (top):
  bat_detective_batdetect2: 1702
  echobank_batdetect2: 27
  bat_data_2018: 22
  bct_1_sec: 12
  bat_data_2018_test: 11
  bat_data_2019: 11
  rhinolophus_bct: 10
  sn_scot_nor: 9
  bcireland: 6

Target no-event count (to reach TOTAL=7379): 1625

FINAL COUNTS:
  Universe TOTAL: 7379 (expected 7379)
  TEST:           369 (expected 369)
  TRAIN:          7010 (expected 7010)
  Dropped no-event (excess): 185

DONE: c

In [9]:
#@title Only run this code if you need to install perch-hoplite
!pip install git+https://github.com/google-research/perch-hoplite.git

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
#@title If you plan to use Perch V2, you must install this version (or later) of TensorFlow and cuda
!pip install tensorflow[and-cuda]~=2.20.0rc0

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
# @title Imports
from etils import epath
from IPython.display import display
import ipywidgets as widgets
import numpy as np
from perch_hoplite.agile import colab_utils
from perch_hoplite.agile import embed
from perch_hoplite.agile import source_info
from perch_hoplite.db import brutalism
from perch_hoplite.db import interface

In [2]:
import tensorflow

# Embed the audio data

In [14]:
# @title Configuration { vertical-output: true }

# @markdown Configure the raw dataset and output location(s).  The format is a mapping from
# @markdown a dataset_name to a (base_path, fileglob) pair.  Note that the file
# @markdown globs are case sensitive.  The dataset name can be anything you want.
#
# @markdown This structure allows you to move your data around without having to
# @markdown re-embed the dataset.  The generated embedding database will be
# @markdown placed in the base path. This allows you to simply swap out
# @markdown the base path here if you ever move your dataset.

# @markdown By default we only process one dataset at a time.  Re-run this entire notebook
# @markdown once per dataset.

# @markdown For example, we might set dataset_base_path to '/home/me/myproject',
# @markdown and use the glob '\*/\*.wav' if all of the audio files have filepaths
# @markdown like '/home/me/myproject/site_XYZ/audio_ABC.wav' (e.g. audio files are contained in subfolders of the base directory).

# @markdown 1. Create a unique name for the database that will store the embeddings for the target data.
dataset_name = 'UK_train_embeddings'  # @param {type:'string'}
# @markdown 2. Input the filepath for the folder that is containing the input audio files.
dataset_base_path = 'C:/Users/Administrator/hello/Yucatan/acoustic_data/data/yucatan/audio_timeexp10'  #@param {type:'string'}
# @markdown 3. Input the file pattern for the audio files within that folder that you want to embed. Some examples for how to input:
# @markdown - All files in the base directory of a specific type (not subdirectories): e.g. `*.wav` (or `*.flac` etc) will generate embeddings for all .wav files (or whichever format) in the dataset_base_path
# @markdown - All files in one level of subdirectories within the base directory: `*/*.flac` will generate embeddings for all .flac files
# @markdown - Single file: `myfile.wav` will only embed the audio from that specific file.
dataset_fileglob = '*.wav'  # @param {type:'string'}

# @markdown 4. [Optional] If saving the embeddings database to a new directory, specify here.
# @markdown Otherwise, leave blank - by default the embeddings database output will be saved within
# @markdown dataset_base_path where the audio is located. You do not need to specify db_path unless you want to maintain multiple
# @markdown distinct embedding databases, or if you would like to save the output
# @markdown in a different folder. If your input audio data is accessed
# @markdown from a public URL, we recommend specifying a separate output directory here.
db_path = ''  # @param {type:'string'}
if not db_path or db_path == 'None':
  db_path = 'C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings'

# @markdown 5. Choose a supported model to generate embeddings: `perch_v2` is the latest Perch model and was trained
# @markdown on multiple taxa include birds, mammals, insects and amphibians; `perch_v2` has also demonstrated high performance for marine audio transfer learning tasks. **NOTE: `perch_v2` only works with GPU runtimes - see above instructions.**
# @markdown `perch_8` is the last updated version of Perch V1 trained only on birds, and `birdnet_v2.3` is also a common option
# @markdown for birds. Other choices include `surfperch` for coral reefs or
# @markdown `multispecies_whale` for marine mammals.
model_choice = 'perch_8'  #@param['perch_v2','perch_8', 'humpback', 'multispecies_whale', 'surfperch', 'birdnet_V2.3']

# @markdown 6. [Optional] Shard the audio for embeddings. File sharding automatically splits audio files into smaller chunks
# @markdown for creating embeddings. This limits both system and GPU memory usage,
# @markdown especially useful when working with long files (>1 hour).
use_file_sharding = True  # @param {type:'boolean'}
# @markdown If you want to change the length in seconds for the shards, specify here.
shard_length_in_seconds = 60  # @param {type:'number'}

audio_glob = source_info.AudioSourceConfig(
    dataset_name=dataset_name,
    base_path=dataset_base_path,
    file_glob=dataset_fileglob,
    min_audio_len_s=1.0,
    target_sample_rate_hz=-2,
    shard_len_s=float(shard_length_in_seconds) if use_file_sharding else None,
)

configs = colab_utils.load_configs(
    source_info.AudioSources((audio_glob,)),
    db_path,
    model_config_key=model_choice,
    db_key='sqlite_usearch',
)
configs

AgileConfigs(audio_sources_config=AudioSources(audio_globs=(AudioSourceConfig(dataset_name='UK_train_embeddings', base_path='C:/Users/Administrator/hello/Yucatan/acoustic_data/data/yucatan/audio_timeexp10', file_glob='*.wav', min_audio_len_s=1.0, target_sample_rate_hz=-2, shard_len_s=60.0, max_shards_per_file=None),)), db_config=DBConfig(db_key='sqlite_usearch', db_config=db_path: C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings
usearch_cfg:
  dtype: float16
  embedding_dim: 1280
  expansion_add: 256
  expansion_search: 128
  metric_name: IP
), model_config=ModelConfig(model_key='taxonomy_model_tf', embedding_dim=1280, model_config=hop_size_s: 5.0
model_path: ''
sample_rate: 32000
tfhub_path: google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier
tfhub_version: 8
window_size_s: 5.0
, logits_key=None, logits_idxes=None))

In [19]:
#@title Initialize the hoplite database (DB) { vertical-output: true }
global db
db = configs.db_config.load_db()
num_embeddings = db.count_embeddings()

print('Initialized DB located at ', configs.db_config.db_config.db_path)

def drop_and_reload_db(_) -> interface.HopliteDBInterface:
  db_path = epath.Path(configs.db_config.db_config.db_path)
  for fp in db_path.glob('hoplite.sqlite*'):
    fp.unlink()
  (db_path / 'usearch.index').unlink()
  print('\n Deleted previous db at: ', configs.db_config.db_config.db_path)
  db = configs.db_config.load_db()

#@markdown If `drop_existing_db` set to True, when the database already exists and contains embeddings,
#@markdown then those existing embeddings will be erased. You will be prompted to confirm you wish to delete those existing
#@markdown embeddings. If you want to keep existing embeddings in the database, then set to False, which will append the new
#@markdown embeddings to the database.
drop_existing_db = False  #@param {type:'boolean'}

if num_embeddings > 0 and drop_existing_db:
  print('Existing DB contains datasets: ', db.get_dataset_names())
  print('num embeddings: ', num_embeddings)
  print('\n\nClick the button below to confirm you really want to drop the database at ')
  print(f'{configs.db_config.db_config.db_path}\n')
  print(f'This will permanently delete all {num_embeddings} embeddings from the existing database.\n')
  print('If you do NOT want to delete this data, set `drop_existing_db` above to `False` and re-run this cell.\n')

  button = widgets.Button(description='Delete database?')
  button.on_click(drop_and_reload_db)
  display(button)

Initialized DB located at  C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings


In [16]:
#@title Run the embedding { vertical-output: true }

print(f'Embedding dataset: {audio_glob.dataset_name}')

worker = embed.EmbedWorker(
    audio_sources=configs.audio_sources_config,
    db=db,
    model_config=configs.model_config)

worker.process_all(target_dataset_name=audio_glob.dataset_name)

print('\n\nEmbedding complete, total embeddings: ', db.count_embeddings())

Embedding dataset: UK_train_embeddings


100%|██████████████████████████████████████████████████████████████████████████████| 1395/1395 [16:35<00:00,  1.40it/s]




Embedding complete, total embeddings:  2630


In [17]:
#@title Per dataset statistics { vertical-output: true }

for dataset in db.get_dataset_names():
  print(f'\nDataset \'{dataset}\':')
  print('\tnum embeddings: ', db.get_embeddings_by_source(dataset, source_id=None).shape[0])


Dataset 'UK_train_embeddings':
	num embeddings:  2630


In [18]:
#@title Show example embedding search
#@markdown As an example (and to show that the embedding process worked), this
#@markdown selects a single embedding from the database and outputs the embedding ids of the
#@markdown top-K (k = 128) nearest neighbors in the database.

q = db.get_embedding(db.get_one_embedding_id())
%time results, scores = brutalism.brute_search(worker.db, query_embedding=q, search_list_size=128, score_fn=np.dot)
print([int(r.embedding_id) for r in results])

CPU times: total: 62.5 ms
Wall time: 500 ms
[1, 12, 3, 208, 231, 105, 1264, 237, 1558, 1135, 238, 2, 239, 1397, 1214, 206, 494, 1228, 165, 1226, 1217, 60, 803, 1320, 106, 149, 150, 1122, 11, 4, 1321, 1201, 87, 810, 1265, 1267, 1307, 1553, 1308, 1202, 59, 159, 967, 1569, 2455, 1560, 207, 1136, 74, 1544, 1133, 2440, 1552, 38, 1227, 1471, 1272, 232, 1271, 1473, 240, 2400, 1472, 152, 171, 770, 125, 61, 811, 2570, 1566, 1274, 1268, 1555, 1210, 75, 1561, 205, 126, 1298, 499, 96, 2571, 534, 446, 2569, 112, 1218, 1215, 496, 249, 127, 166, 1170, 84, 965, 815, 2439, 1263, 1562, 807, 776, 506, 1213, 1304, 769, 1097, 448, 172, 1134, 136, 1211, 1252, 1310, 1231, 91, 1305, 806, 1299, 1273, 1469, 70, 175, 1095, 1148, 766, 137, 2562]


In [1]:
!jupyter nbconvert 1.2_embed_audio_UK.ipynb --to markdown --ClearOutputPreprocessor.enabled=True

[NbConvertApp] Converting notebook 1.2_embed_audio_UK.ipynb to markdown
[NbConvertApp] Writing 10372 bytes to 1.2_embed_audio_UK.md
